# 03 — Multi-File Binary Classification

Pools all 8 CICIDS2017 files and runs an 80/20 stratified binary classification experiment.

In [8]:
%run 00_data_loader.ipynb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

Config loaded
preprocess_binary() defined
load_all_datasets() defined


## 1. Load All 8 Datasets

In [9]:
datasets = load_all_datasets()
print(f"Loaded: {list(datasets.keys())}")

Loading 8 dataset(s): ['monday', 'bruteforce', 'dos', 'web_attacks', 'infiltration', 'botnet', 'portscan', 'ddos']

  [monday]  Monday-WorkingHours.pcap_ISCX.csv
           raw (529918, 79)  ->  clean (502650, 81)  (attack rate 0.0%)
  [bruteforce]  Tuesday-WorkingHours.pcap_ISCX.csv
           raw (445909, 79)  ->  clean (421626, 81)  (attack rate 2.2%)
  [dos]  Wednesday-workingHours.pcap_ISCX.csv
           raw (692703, 79)  ->  clean (610492, 81)  (attack rate 31.7%)
  [web_attacks]  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
           raw (170366, 79)  ->  clean (164179, 81)  (attack rate 1.3%)
  [infiltration]  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
           raw (288602, 79)  ->  clean (252790, 81)  (attack rate 0.0%)
  [botnet]  Friday-WorkingHours-Morning.pcap_ISCX.csv
           raw (191033, 79)  ->  clean (184044, 81)  (attack rate 1.1%)
  [portscan]  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
           raw (286467, 79)  ->  clea

## 2. Pool and Split

In [10]:
combined = pd.concat(datasets.values(), ignore_index=True)
print("Combined shape:", combined.shape)
print("\nRows per source file:")
print(combined["Source_File"].value_counts())
print("\nBinary label distribution:")
print(combined["Label_Binary"].value_counts())

Combined shape: (2572640, 81)

Rows per source file:
Source_File
DoS             610492
Monday          502650
BruteForce      421626
Infiltration    252790
DDoS            223082
PortScan        213777
Botnet          184044
WebAttacks      164179
Name: count, dtype: int64

Binary label distribution:
Label_Binary
0    2146899
1     425741
Name: count, dtype: int64


In [11]:
X = combined.drop(columns=["Label", "Label_Binary", "Source_File"])
y = combined["Label_Binary"]

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print("Train label counts:\n", y_train.value_counts())

X_train: (2058112, 78)   X_test: (514528, 78)
Train label counts:
 Label_Binary
0    1717519
1     340593
Name: count, dtype: int64


## 3. Logistic Regression

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr   = log_reg.predict(X_test_scaled)
y_scores_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression")
print("  Accuracy: ", round(accuracy_score(y_test, y_pred_lr), 4))
print("  Precision:", round(precision_score(y_test, y_pred_lr), 4))
print("  Recall:   ", round(recall_score(y_test, y_pred_lr), 4))
print("  F1:       ", round(f1_score(y_test, y_pred_lr), 4))
print("  ROC-AUC:  ", round(roc_auc_score(y_test, y_scores_lr), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

Logistic Regression
  Accuracy:  0.9404
  Precision: 0.744
  Recall:    0.976
  F1:        0.8443
  ROC-AUC:   0.9883

Confusion Matrix:
 [[400781  28599]
 [  2045  83103]]


## 4. Random Forest

In [13]:
rf_model = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42
)
rf_model.fit(X_train, y_train)
y_pred_rf   = rf_model.predict(X_test)
y_scores_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest")
print("  Accuracy: ", round(accuracy_score(y_test, y_pred_rf), 4))
print("  Precision:", round(precision_score(y_test, y_pred_rf), 4))
print("  Recall:   ", round(recall_score(y_test, y_pred_rf), 4))
print("  F1:       ", round(f1_score(y_test, y_pred_rf), 4))
print("  ROC-AUC:  ", round(roc_auc_score(y_test, y_scores_rf), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

Random Forest
  Accuracy:  0.9986
  Precision: 0.9966
  Recall:    0.995
  F1:        0.9958
  ROC-AUC:   0.9998

Confusion Matrix:
 [[429087    293]
 [   425  84723]]


## 5. Results Summary

In [14]:
pd.DataFrame([
    {"Model": "Logistic Regression",
     "Accuracy": accuracy_score(y_test, y_pred_lr), "Precision": precision_score(y_test, y_pred_lr),
     "Recall": recall_score(y_test, y_pred_lr), "F1": f1_score(y_test, y_pred_lr),
     "ROC_AUC": roc_auc_score(y_test, y_scores_lr)},
    {"Model": "Random Forest",
     "Accuracy": accuracy_score(y_test, y_pred_rf), "Precision": precision_score(y_test, y_pred_rf),
     "Recall": recall_score(y_test, y_pred_rf), "F1": f1_score(y_test, y_pred_rf),
     "ROC_AUC": roc_auc_score(y_test, y_scores_rf)},
]).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.9404,0.7440,0.976,0.8443,0.9883
1,Random Forest,0.9986,0.9966,0.995,0.9958,0.9998
